In [ ]:
import os
from getpass import getpass

# --- Project parameters ---
repo_path = "/content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject"
github_email = "s309629@studenti.polito.it"
github_username = "AndreaPassera"
repo_url = "https://github.com/AndreaPassera/MaskArchitectureAnomaly_CourseProject.git"
# ------------------------------

print("Moving to the project directory...")
os.chdir(repo_path)

print("Configuring Git variables...")
os.system(f'git config --global user.email "{github_email}"')
os.system(f'git config --global user.name "{github_username}"')

# Request token for HTTPS authentication
token = getpass("Paste your GitHub token (PAT) here: ")

# Rebuild the URL with embedded authentication
auth_url = repo_url.replace("https://", f"https://{github_username}:{token}@")

print("\nAdding files to the index (git add .)...")
os.system("git add .")

commit_msg = input("Enter the commit message: ")
os.system(f'git commit -m "{commit_msg}"')

print("\nPushing to the remote repository...")
# Temporarily replace the remote to push with the token
os.system("git remote remove origin")
os.system(f"git remote add origin {auth_url}")

# --- FIX PER I BRANCH ---
# Rinomina forzatamente il ramo locale corrente in 'main'
os.system("git branch -M main")

# Fai il push forzando l'aggiornamento del ramo 'main' remoto
exit_code = os.system("git push -u origin main")

if exit_code != 0:
    print("\nWARNING: Push failed. Check your token or internet connection.")
else:
    print("\nSUCCESS: Files successfully pushed to the 'main' branch!")

# Restore the clean remote URL (without token) to keep the workspace safe
os.system("git remote remove origin")
os.system(f"git remote add origin {repo_url}")

print("\nOperation completed.")

In [ ]:
import os
import subprocess
from getpass import getpass

# --- Parametri del progetto ---
repo_path = "/content/drive/MyDrive/FAIMDL Project/MaskArchitectureAnomaly_CourseProject"
github_email = "s309629@studenti.polito.it"  # INSERISCI LA TUA EMAIL
github_username = "AndreaPassera"
repo_url = "https://github.com/AndreaPassera/MaskArchitectureAnomaly_CourseProject.git"

def run_git(command, hide_error=False):
    """Esegue un comando git e stampa il risultato vero."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0 and not hide_error:
        print(f"\n❌ ERRORE NEL COMANDO: {command}")
        print(f"Dettaglio Errore Git:\n{result.stderr}")
    return result.returncode, result.stdout, result.stderr

print("1. Monto Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

print("\n2. Spostamento nella cartella del progetto...")
os.chdir(repo_path)

print("3. Inizializzazione Git (se necessaria)...")
run_git("git init") # Se è già inizializzata, non fa danni

print("4. Configurazione Git...")
run_git(f'git config --global user.email "{github_email}"')
run_git(f'git config --global user.name "{github_username}"')

# Richiesta Token
token = getpass("\n🔑 Incolla il token di GitHub (PAT): ")
auth_url = repo_url.replace("https://", f"https://{github_username}:{token}@")

print("\n5. Aggiunta file all'indice...")
run_git("git add .")

print("\n6. Creazione Commit...")
commit_msg = input("Inserisci il messaggio di commit: ")
run_git(f'git commit -m "{commit_msg}"', hide_error=True) # Nascondo errore se non ci sono modifiche

print("\n7. Configurazione Remote...")
run_git("git remote remove origin", hide_error=True)
run_git(f"git remote add origin {auth_url}")
run_git("git branch -M main")

print("\n8. TENTATIVO DI PUSH (Vediamo il vero errore se fallisce)...")
code, out, err = run_git("git push -u origin main")

if code == 0:
    print("\n✅ SUCCESS: Push completato! I file sono su GitHub.")
else:
    # Se fallisce, analizziamo l'errore per capire se è un conflitto (fetch origin)
    if "fetch first" in err or "non-fast-forward" in err:
        print("\n⚠️ CONFLITTO RILEVATO: Su GitHub ci sono file che tu non hai (es. README).")
        risposta = input("Vuoi forzare il push (ATTENZIONE: cancellerà i file su GitHub che non hai su Drive)? (y/n): ")
        if risposta.lower() == 'y':
            print("Forzatura in corso...")
            run_git("git push -u origin main --force")
            print("✅ SUCCESS: Push forzato completato.")
        else:
            print("🛑 Operazione annullata. Devi scaricare prima le modifiche con 'git pull'.")

# Pulizia di sicurezza
run_git("git remote remove origin", hide_error=True)
run_git(f"git remote add origin {repo_url}")
print("\nOperazione terminata.")